# 05 - Geography tags, then a dashboard

Goal: tag every trace with the customer's country and region before export, then use the tags to slice traffic in LangSmith.

OTel traces carry nothing about where the customer is. Phase 1 users tag and filter in the UI. Phase 2 puts the tag in the pipeline: the same processor from challenge 3, now given a geography lookup, so the tag is on every trace the same way.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

## Step 1: A geography source

Where a customer is comes from your systems: an edge header, a tenant record, a CRM region. `bank_traffic.GEO_REGISTRY` stands in for that lookup.

In [ ]:
bank_traffic.GEO_REGISTRY.lookup("example-conversation-id")

## Step 2: Write `geography_attributes`

It receives a conversation id and returns **span attributes**: `langsmith.metadata.customer_country` and `langsmith.metadata.customer_region` from the registry (use `otel_setup.to_langsmith_attributes`), and `langsmith.span.tags`:
a list with `region:<REGION>` and `country:<CC>`. Metadata is for filtering and grouping; a tag is the quick filter chip in the trace list.

In [ ]:
# Reference solution: try writing this yourself first.
def geography_attributes(conversation_id):
    """Challenge 5: where the customer is, as span attributes and tags."""
    geo = bank_traffic.GEO_REGISTRY.lookup(conversation_id)
    attrs = otel_setup.to_langsmith_attributes(geo)
    attrs["langsmith.span.tags"] = [f"region:{geo['customer_region']}", f"country:{geo['customer_country']}"]
    return attrs

In [ ]:
geography_attributes("example-conversation-id")

## Step 3: Add it to the pipeline, and run

`ref.GeographyProcessor` is the challenge 3 processor plus a call to your function for the trace's conversation id. Sixteen turns give the dashboard something to split.

In [ ]:
otel_setup.configure(resource_attributes=ref.resource_attributes(),
                     processors=[ref.GeographyProcessor(geography_attributes)])
agent = bank_agent.build_agent(risk_hook=ref.risk_metadata, tool_hook=ref.tool_metadata)
bank_traffic.run_turns(bank_agent, agent, (bank_traffic.QUESTIONS + bank_traffic.QUESTIONS[:4]), per_request=ref.per_request)
otel_setup.flush()
n = bank_traffic.wait_for_runs(PROJECT, minimum=20)
print(f"{n} root runs in {PROJECT}")

## Step 4: Traffic by geography, and a metadata query

The grouping a dashboard does, in code. Then the same keys used as a LangSmith filter.

In [ ]:
import pandas as pd
time.sleep(10)
roots = [r for r in enrich.project_runs(client, PROJECT, hours=1) if r.name == "bank_support_turn"]
rows = []
for r in roots:
    m = enrich.meta_of(r)
    rows.append({"region": m.get("customer_region"), "country": m.get("customer_country"), "error": bool(r.error)})
df = pd.DataFrame(rows)
display(df.groupby(["region", "country"], dropna=False).agg(traces=("error", "size"), errors=("error", "sum")))

f = 'and(eq(metadata_key, "customer_region"), eq(metadata_value, "APAC"))'
print("APAC runs via filter:", len(list(client.list_runs(project_name=PROJECT, filter=f, limit=100))))

## Step 5: Build the dashboard in LangSmith

Open your project (link below) and:

1. Filter the trace list by tag `region:APAC`, or by `metadata.customer_country = SG`.
2. Open the **Monitor** tab. Choose **Group by > Metadata** and pick `customer_region`: trace count, latency and error rate split by region.
3. To keep it: create a **custom dashboard**, add *Trace count* grouped by metadata `customer_region`, and *Error rate* grouped by `customer_country`.

If the `region:` tag does not show in the trace list, the metadata still works: the tag is set through the `langsmith.span.tags` attribute, and whether your LangSmith version reads it is not guaranteed.

In [ ]:
lab_links.project_link(hint="Monitor tab > Group by metadata > customer_region; then save the charts to a custom dashboard")

**Result:** a table of traces by region and country, and a dashboard in LangSmith showing the same split.

Next: challenge 6 curates a golden dataset from the best of this traffic.